### [Day 18: Operation Order](https://adventofcode.com/2020/day/18)

As you look out the window and notice a heavily-forested continent slowly appear over the horizon, you are interrupted by the child sitting next to you. They're curious if you could help them with their math homework.

 Unfortunately, it seems like this "math" [follows different rules](https://www.youtube.com/watch?v=3QtRK7Y2pPU&t=15) than you remember.

 The homework (your puzzle input) consists of a series of expressions that consist of addition (`+`), multiplication (`*`), and parentheses (`(...)`). Just like normal math, parentheses indicate that the expression inside must be evaluated before it can be used by the surrounding expression. Addition still finds the sum of the numbers on both sides of the operator, and multiplication still finds the product.

 However, the rules of **operator precedence** have changed. Rather than evaluating multiplication before addition, the operators have the **same precedence**, and are evaluated left-to-right regardless of the order in which they appear.

 For example, the steps to evaluate the expression `1 + 2 * 3 + 4 * 5 + 6` are as follows:

 
```
1 + 2 * 3 + 4 * 5 + 6
  3   * 3 + 4 * 5 + 6
      9   + 4 * 5 + 6
         13   * 5 + 6
             65   + 6
                 71
```

 Parentheses can override this order; for example, here is what happens if parentheses are added to form `1 + (2 * 3) + (4 * (5 + 6))`:

 
```
1 + (2 * 3) + (4 * (5 + 6))
1 +    6    + (4 * (5 + 6))
     7      + (4 * (5 + 6))
     7      + (4 *   11   )
     7      +     44
            51
```

 Here are a few more examples:

  
  - `2 * 3 + (4 * 5)` becomes **`26`**. 
  - `5 + (8 * 3 + 9 + 3 * 4 * 3)` becomes **`437`**. 
  - `5 * 9 * (7 * 3 * 3 + 9 * 3 + (8 + 6 * 4))` becomes **`12240`**. 
  - `((2 + 4 * 9) * (6 + 9 * 8 + 6) + 6) + 2 + 4 * 2` becomes **`13632`**. 
  
Before you can help with the homework, you need to understand it yourself. **Evaluate the expression on each line of the homework; what is the sum of the resulting values?**

In [1]:
with open("2020/data/advent_18.txt") as f:
    lines = [line.strip() for line in f if line.strip()]

In [2]:
import re

def get_parentheses_content(line):
    start = None
    counter = 0
    for i, char in enumerate(line):
        if char == '(':
            if counter == 0:
                # first open
                start = i
            counter += 1
        elif char == ')':
            counter -= 1
            if counter == 0:
                return line[start+1:i], start, i

    raise ValueError(f"This should not happen")

def calc(line):
    while '(' in line:
        subexpr, start, end = get_parentheses_content(line)
        subresult = calc(subexpr)
        line = line[:start] + str(subresult) + line[end+1:]

    tokens = line.strip().split(' ')
    
    total = int(tokens[0])
    i = 1
    while i < len(tokens):
        op = tokens[i]
        arg = int(tokens[i+1])
        if op == '+':
            total += arg
        elif op == '*':
            total *= arg
        else:
            raise ValueError(f"Unknown operator: {op}")
        i += 2

    return total

In [3]:
print(f'The sum of the resulting values is {sum([calc(line) for line in lines])}')

The sum of the resulting values is 24650385570008


### Part two

You manage to answer the child's questions and they finish part 1 of their homework, but get stuck when they reach the next section: **advanced** math.

 Now, addition and multiplication have **different** precedence levels, but they're not the ones you're familiar with. Instead, addition is evaluated **before** multiplication.

 For example, the steps to evaluate the expression `1 + 2 * 3 + 4 * 5 + 6` are now as follows:

 
```
1 + 2 * 3 + 4 * 5 + 6
  3   * 3 + 4 * 5 + 6
  3   *   7   * 5 + 6
  3   *   7   *  11
     21       *  11
         231
```

 Here are the other examples from above:

  
  - `1 + (2 * 3) + (4 * (5 + 6))` still becomes **`51`**. 
  - `2 * 3 + (4 * 5)` becomes **`46`**. 
  - `5 + (8 * 3 + 9 + 3 * 4 * 3)` becomes **`1445`**. 
  - `5 * 9 * (7 * 3 * 3 + 9 * 3 + (8 + 6 * 4))` becomes **`669060`**. 
  - `((2 + 4 * 9) * (6 + 9 * 8 + 6) + 6) + 2 + 4 * 2` becomes **`23340`**.  
  
**What do you get if you add up the results of evaluating the homework problems using these new rules?**

In [4]:
def calc2(line):
    while '(' in line:
        subexpr, start, end = get_parentheses_content(line)
        subresult = calc2(subexpr)
        line = line[:start] + str(subresult) + line[end+1:]

    tokens = line.strip().split(' ')
    
    # Evaluate additions first
    tokens_cp = tokens[:]
    i = 0
    while i < len(tokens_cp):
        if tokens_cp[i] == '+':
            lhs = int(tokens_cp[i-1])
            rhs = int(tokens_cp[i+1])
            result = lhs + rhs
            tokens_cp = tokens_cp[:i-1] + [str(result)] + tokens_cp[i+2:]
            i = 0  
        else:
            i += 1

    # Evaluate multiplications
    total = int(tokens_cp[0])
    i = 1
    while i < len(tokens_cp):
        op = tokens_cp[i]
        arg = int(tokens_cp[i+1])
        if op == '*':
            total *= arg
        else:
            raise ValueError(f"Unknown operator after + pass: {op}")
        i += 2

    return total

In [5]:
print(f'The sum of the resulting values is {sum([calc2(line) for line in lines])}')

The sum of the resulting values is 158183007916215
